# Expected SARSA on the original long route

Compare adaptive sensing against learned all-on, fixed driving, and random driving on the 2,250 m / 120 s route. Run All reads the bundled results. Set `RUN_TRAINING = True` for a fresh run.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("SDL_AUDIODRIVER", "dummy")
os.environ.setdefault("PYGAME_HIDE_SUPPORT_PROMPT", "1")
here = Path.cwd().resolve()
repo = next((p for p in (here, *here.parents) if (p / "runs/index.csv").is_file()), None)
if repo is None:
    raise RuntimeError("Open this notebook from the MORL4AVSO checkout.")
sys.path.insert(0, str(repo))


import json

import pandas as pd
from IPython.display import display

from environments.sensor_highway import ExperimentConfig
from experiments.route_notebook_helpers import (
    evaluate_route, new_run_directory, summarize_trials, train_sarsa_pair,
)

In [ ]:
RUN_TRAINING = False
TRAIN_EPISODES = None  # None uses the recorded episode budget.
recorded = repo / "runs/training/time_completion_120s_1000ep_20261121_204241_241658"

config = ExperimentConfig.load(recorded / "config.json")
metadata = json.loads((recorded / "metadata.json").read_text())
protocol = {key: metadata[key] for key in ("settings", "training_seed", "test_seeds", "test_weather")}
if TRAIN_EPISODES is not None:
    protocol["settings"]["episodes"] = TRAIN_EPISODES

## Matched evaluation

Finish time averages successful trips. Sensing power averages all trips.

In [ ]:
if RUN_TRAINING:
    output = new_run_directory(repo, "sarsa_long_route")
    agents = train_sarsa_pair(config, protocol, output)
    agents.update({"idle_all_on": None, "random_all_on": None})
    trials = evaluate_route(config, agents, protocol)
    trials.to_csv(output / "evaluation.csv", index=False)
    summarize_trials(trials).to_csv(output / "summary.csv")
    print(f"Saved to {output.relative_to(repo)}")
else:
    trials = pd.read_csv(recorded / "evaluation.csv")

display(summarize_trials(trials).round(3))